# Tétel 01 — Többdimenziós normális & normális korreláció tétele

**Cél:** Kézzel, csak numpy-jal megérteni mi történik a mátrixok szintjén.  
Generálunk random adatokat, kiszámoljuk a várhatóérték-vektort, a kovarianciamátrixot, majd szemléltetjük a **normális korreláció tételét** számokkal.

---
Jelölések (mint a tételben):
- $\underline{\xi} \in \mathbb{R}^2$ — amit "becsülni" akarunk (2 változó)
- $\underline{\eta} \in \mathbb{R}^1$ — amit "ismerünk" (1 változó)
- Együttesen: $(\underline{\xi}, \underline{\eta}) \sim N_3(\underline{\mu}, \Sigma)$

In [11]:
import numpy as np
import pandas as pd  # csak Data Wranglerhez a végén

np.random.seed(42)
print("Kész. Csak numpy + pandas (csak megjelenítéshez).")

Kész. Csak numpy + pandas (csak megjelenítéshez).


## 1. Adatok generálása

Megadjuk a "valódi" paramétereket ($\underline{\mu}$, $\Sigma$), és Cholesky-felbontással generálunk mintát.

**Cholesky trükk:** ha $Z \sim N(0, I)$ és $\Sigma = LL^T$, akkor $LZ + \mu \sim N(\mu, \Sigma)$.

In [12]:
n = 300   # megfigyelések száma
p = 3     # dimenziók összesen: xi_1, xi_2, eta

# --- Valódi paraméterek (ezeket akarjuk majd visszabecsülni) ---
mu_igazi = np.array([2.0, -1.0, 5.0])

# Kovarianciamátrix (szimmetrikus, pozitív definit)
# Sorban: xi_1, xi_2, eta
Sigma_igazi = np.array([
    [4.0, 1.5, 2.0],   # xi_1 sora
    [1.5, 2.0, 0.8],   # xi_2 sora
    [2.0, 0.8, 3.0]    # eta sora
])

print("Valódi mu:")
print(mu_igazi)
print("\nValódi Sigma:")
print(Sigma_igazi)

Valódi mu:
[ 2. -1.  5.]

Valódi Sigma:
[[4.  1.5 2. ]
 [1.5 2.  0.8]
 [2.  0.8 3. ]]


In [13]:
# Cholesky: Sigma = L @ L.T
L = np.linalg.cholesky(Sigma_igazi)
print("L (alsó háromszög, Sigma = L @ L.T):")
print(L.round(4))

# Ellenőrzés: L @ L.T == Sigma_igazi?
print("\nL @ L.T (kell == Sigma_igazi):")
print((L @ L.T).round(4))

L (alsó háromszög, Sigma = L @ L.T):
[[2.     0.     0.    ]
 [0.75   1.199  0.    ]
 [1.     0.0417 1.4136]]

L @ L.T (kell == Sigma_igazi):
[[4.  1.5 2. ]
 [1.5 2.  0.8]
 [2.  0.8 3. ]]


In [14]:
# Generálás: Z_raw ~ N(0, I_3), majd transzformáljuk
Z_raw = np.random.randn(n, p)       # n x 3 standard normális
Z = Z_raw @ L.T + mu_igazi          # n x 3 -- most már N(mu, Sigma)

print(f"Adatmátrix Z mérete: {Z.shape}  ({n} sor = megfigyelés, {p} oszlop = változó)")
print("\nElső 5 sor (xi_1, xi_2, eta):")
print(Z[:5].round(3))

Adatmátrix Z mérete: (300, 3)  (300 sor = megfigyelés, 3 oszlop = változó)

Első 5 sor (xi_1, xi_2, eta):
[[ 2.993 -0.793  6.407]
 [ 5.046 -0.138  6.182]
 [ 5.158  1.105  5.948]
 [ 3.085 -1.149  4.865]
 [ 2.484 -3.112  2.724]]


## 2. Várhatóérték-vektor kézzel

$$\hat{\underline{\mu}} = \frac{1}{n} \sum_{i=1}^n \underline{z}_i = \frac{1}{n} Z^T \mathbf{1}_n$$

Mátrixosan: összegezzük az összes sort, és elosztjuk $n$-nel.

In [15]:
# Kézzel: oszloponkénti összeg / n
mu_hat = np.zeros(p)
for i in range(n):
    mu_hat = mu_hat + Z[i]   # összeadjuk az összes sort
mu_hat = mu_hat / n

print("Becsült mu_hat (kézzel, for-ciklussal):")
print(mu_hat.round(4))

print("\nValódi mu_igazi:")
print(mu_igazi)

# Ellenőrzés numpy-jal
print("\nEllenőrzés: Z.mean(axis=0):")
print(Z.mean(axis=0).round(4))   # axis=0 = sorok mentén átlagol -> 1 sor marad

Becsült mu_hat (kézzel, for-ciklussal):
[ 2.134  -1.0381  5.1514]

Valódi mu_igazi:
[ 2. -1.  5.]

Ellenőrzés: Z.mean(axis=0):
[ 2.134  -1.0381  5.1514]


## 3. Kovarianciamátrix kézzel

Először **centráljuk** az adatokat (kivonjuk a várhatóértéket):
$$\tilde{Z} = Z - \mathbf{1}_n \hat{\underline{\mu}}^T$$

Majd a kovarianciamátrix:
$$\hat{\Sigma} = \frac{1}{n-1} \tilde{Z}^T \tilde{Z}$$

Ez a $p \times p$-es mátrix. Az $(r,s)$-edik eleme az $r$-edik és $s$-edik változó kovarianciája.

In [16]:
# Centrálás: minden sorból kivonjuk a mu_hat-ot
Z_cent = Z - mu_hat   # broadcasting: mu_hat (3,) automatikusan minden sorból kivonódik

print("Z_cent első 5 sora (centált adatok):")
print(Z_cent[:5].round(3))
print("\nEllenőrzés: Z_cent oszlopátlagai (kell ~0):")
print(Z_cent.mean(axis=0).round(6))

Z_cent első 5 sora (centált adatok):
[[ 0.859  0.245  1.255]
 [ 2.912  0.9    1.031]
 [ 3.024  2.143  0.796]
 [ 0.951 -0.111 -0.287]
 [ 0.35  -2.074 -2.428]]

Ellenőrzés: Z_cent oszlopátlagai (kell ~0):
[-0. -0.  0.]


In [17]:
# Kovarianciamátrix: (Z_cent.T @ Z_cent) / (n-1)
S = (Z_cent.T @ Z_cent) / (n - 1)   # (3x300) @ (300x3) -> 3x3

print("Becsült kovarianciamátrix S = Z_cent.T @ Z_cent / (n-1):")
print(S.round(4))

print("\nValódi Sigma_igazi:")
print(Sigma_igazi)

print("\nEllenőrzés: np.cov(Z.T):")
print(np.cov(Z.T).round(4))

Becsült kovarianciamátrix S = Z_cent.T @ Z_cent / (n-1):
[[3.202  1.2797 1.4917]
 [1.2797 1.9573 0.5637]
 [1.4917 0.5637 2.8302]]

Valódi Sigma_igazi:
[[4.  1.5 2. ]
 [1.5 2.  0.8]
 [2.  0.8 3. ]]

Ellenőrzés: np.cov(Z.T):
[[3.202  1.2797 1.4917]
 [1.2797 1.9573 0.5637]
 [1.4917 0.5637 2.8302]]


## 4. Blokk-struktúra

A tételben a kovarianciamátrix **blokk-particionált**:
$$\Sigma = \begin{pmatrix} \Sigma_{\xi\xi} & \Sigma_{\xi\eta} \\ \Sigma_{\eta\xi} & \Sigma_{\eta\eta} \end{pmatrix}$$

- $\Sigma_{\xi\xi}$: $\underline{\xi}$ saját kovarianciamátrixa ($2\times 2$)
- $\Sigma_{\xi\eta}$: kereszt-kovariancia ($2\times 1$) — "mennyire mozognak együtt"
- $\Sigma_{\eta\eta}$: $\underline{\eta}$ varianciája ($1\times 1$)

In [18]:
# Blokkok kiszedése az S mátrixból
S_xixi   = S[0:2, 0:2]   # xi_1, xi_2 egymás közt
S_xieta  = S[0:2, 2:3]   # xi_1, xi_2 és eta között  (2x1)
S_etaeta = S[2:3, 2:3]   # eta önmaga (1x1 = variancia)

print("Σ_ξξ  (xi saját kovarianciamátrixa, 2x2):")
print(S_xixi.round(4))

print("\nΣ_ξη  (xi és eta kereszt-kovariancia, 2x1):")
print(S_xieta.round(4))

print("\nΣ_ηη  (eta varianciája, 1x1):")
print(S_etaeta.round(4))

print("\nValódi értékek összehasonlításhoz:")
print(" Sigma_xixi  =", Sigma_igazi[0:2, 0:2])
print(" Sigma_xieta =", Sigma_igazi[0:2, 2:3].flatten())
print(" Sigma_etaeta=", Sigma_igazi[2, 2])

Σ_ξξ  (xi saját kovarianciamátrixa, 2x2):
[[3.202  1.2797]
 [1.2797 1.9573]]

Σ_ξη  (xi és eta kereszt-kovariancia, 2x1):
[[1.4917]
 [0.5637]]

Σ_ηη  (eta varianciája, 1x1):
[[2.8302]]

Valódi értékek összehasonlításhoz:
 Sigma_xixi  = [[4.  1.5]
 [1.5 2. ]]
 Sigma_xieta = [2.  0.8]
 Sigma_etaeta= 3.0


## 5. Normális korreláció tétele — feltételes várható érték

**Tétel:** Ha $(\underline{\xi}, \underline{\eta})$ együttesen normális, akkor:

$$E[\underline{\xi} \mid \underline{\eta} = \underline{y}] = \underline{\mu}_{\xi} + \Sigma_{\xi\eta} \Sigma_{\eta\eta}^{-1} (\underline{y} - \underline{\mu}_{\eta})$$

Ez **lineáris** $\underline{y}$-ban — mint egy regressziós egyenes, de vektorosan!

**Szemléletes:** ha $\eta$ átlag felett van, ez "húzza" $\xi$-t is — attól függően, mekkora a $\Sigma_{\xi\eta}$ kereszt-kovariancia.

In [19]:
mu_xi  = mu_hat[0:2]   # xi átlaga (2-elem vektor)
mu_eta = mu_hat[2:3]   # eta átlaga (1-elem vektor)

S_etaeta_inv = np.linalg.inv(S_etaeta)

# Regressziós együttható: B = Sigma_xieta @ Sigma_etaeta_inv  (2x1)
B = S_xieta @ S_etaeta_inv
print("Regressziós együttható B = Σ_ξη @ Σ_ηη⁻¹ (2x1):")
print(B.round(4))
print("(Ez mondja meg: ha eta 1-gyel nő, xi_1 mennyivel nő, xi_2 mennyivel nő)")

Regressziós együttható B = Σ_ξη @ Σ_ηη⁻¹ (2x1):
[[0.5271]
 [0.1992]]
(Ez mondja meg: ha eta 1-gyel nő, xi_1 mennyivel nő, xi_2 mennyivel nő)


In [20]:
# E[xi | eta = y] különböző y értékekre
y_ertekek = [3.0, 5.0, 6.0, 8.0]   # az eta igazi átlaga ~5

print(f"mu_xi (átlagos xi) = {mu_xi.round(3)}")
print(f"mu_eta (átlagos eta) = {mu_eta.flatten()[0]:.3f}")
print()
print(f"{'eta értéke':>12}  {'E[xi_1|eta]':>12}  {'E[xi_2|eta]':>12}")
print("-" * 42)

for y in y_ertekek:
    y_vec = np.array([y])
    E_xi = mu_xi + B @ (y_vec - mu_eta)   # 2x1
    print(f"{y:>12.1f}  {E_xi[0]:>12.4f}  {E_xi[1]:>12.4f}")

mu_xi (átlagos xi) = [ 2.134 -1.038]
mu_eta (átlagos eta) = 5.151

  eta értéke   E[xi_1|eta]   E[xi_2|eta]
------------------------------------------
         3.0        1.0000       -1.4667
         5.0        2.0542       -1.0683
         6.0        2.5813       -0.8691
         8.0        3.6354       -0.4708


In [21]:
# Empirikus ellenőrzés: vesszük azokat a sorokat ahol eta ~ y, és megnézzük a tényleges xi átlagot
print("Empirikus ellenőrzés (szűk sáv: |eta - y| < 0.5):")
print(f"{'eta célja':>10}  {'E[xi_1] (képlet)':>18}  {'xi_1 átlag (tényleges)':>22}  {'n db':>6}")
print("-" * 62)

for y in y_ertekek:
    mask = np.abs(Z[:, 2] - y) < 0.5   # bool tömb
    if mask.sum() > 5:
        xi1_atlag = Z[mask, 0].mean()
        y_vec = np.array([y])
        E_xi1 = (mu_xi + B @ (y_vec - mu_eta))[0]
        print(f"{y:>10.1f}  {E_xi1:>18.4f}  {xi1_atlag:>22.4f}  {mask.sum():>6}")

Empirikus ellenőrzés (szűk sáv: |eta - y| < 0.5):
 eta célja    E[xi_1] (képlet)  xi_1 átlag (tényleges)    n db
--------------------------------------------------------------
       3.0              1.0000                  1.1757      42
       5.0              2.0542                  1.9852      73
       6.0              2.5813                  2.6645      61
       8.0              3.6354                  3.9643      12


## 6. Feltételes szórásmátrix — nem függ eta értékétől!

$$\text{Var}[\underline{\xi} \mid \underline{\eta} = \underline{y}] = \Sigma_{\xi\xi} - \Sigma_{\xi\eta} \Sigma_{\eta\eta}^{-1} \Sigma_{\xi\eta}^T$$

**Kulcsüzenet:** nincs $\underline{y}$ a képletben → **konstans**, bárhol is van $\eta$.

In [22]:
S_xi_given_eta = S_xixi - S_xieta @ S_etaeta_inv @ S_xieta.T

print("Feltételes szórásmátrix: Σ_{ξ|η} = Σ_ξξ - Σ_ξη Σ_ηη⁻¹ Σ_ξη^T")
print("(Ez UGYANEZ bármely eta értékre – nincs benne y!)")
print()
print("Σ_{ξ|η} =")
print(S_xi_given_eta.round(4))

print("\nÖsszehasonlítás: Σ_ξξ (feltétel nélküli, nagyobb):")
print(S_xixi.round(4))

print("\nMekkora varianciát 'magyaráz meg' az eta ismerete:")
print((S_xixi - S_xi_given_eta).round(4))

Feltételes szórásmátrix: Σ_{ξ|η} = Σ_ξξ - Σ_ξη Σ_ηη⁻¹ Σ_ξη^T
(Ez UGYANEZ bármely eta értékre – nincs benne y!)

Σ_{ξ|η} =
[[2.4157 0.9826]
 [0.9826 1.8451]]

Összehasonlítás: Σ_ξξ (feltétel nélküli, nagyobb):
[[3.202  1.2797]
 [1.2797 1.9573]]

Mekkora varianciát 'magyaráz meg' az eta ismerete:
[[0.7863 0.2971]
 [0.2971 0.1123]]


In [23]:
# Mennyit csökkent a bizonytalanság? (%-ban)
print("Variancia-csökkentés az eta ismeretének hatására:")
print()
for i in range(2):
    var_before = S_xixi[i, i]
    var_after  = S_xi_given_eta[i, i]
    csökkenes  = (var_before - var_after) / var_before * 100
    print(f"  xi_{i+1}: {var_before:.4f} -> {var_after:.4f}  ({csökkenes:.1f}% csökkent)")

Variancia-csökkentés az eta ismeretének hatására:

  xi_1: 3.2020 -> 2.4157  (24.6% csökkent)
  xi_2: 1.9573 -> 1.8451  (5.7% csökkent)


## 7. Adatok Data Wranglerben

Az összes generált adatot betesszük egy DataFrame-be — kattints a `df` cellára jobb gombbal → **Open in Data Wrangler** (vagy a cella outputján az ikon).

In [24]:
df = pd.DataFrame(Z, columns=['xi_1', 'xi_2', 'eta'])

# Hozzáadjuk a becsült feltételes várhatóértékeket is
eta_vals = df['eta'].values.reshape(-1, 1)
E_xi_all = (mu_xi.reshape(1, -1) + (eta_vals - mu_eta) @ B.T)  # n x 2
df['E_xi1_given_eta'] = E_xi_all[:, 0]
df['E_xi2_given_eta'] = E_xi_all[:, 1]
df['xi1_resid'] = df['xi_1'] - df['E_xi1_given_eta']   # maradéktagok

print(f"DataFrame: {df.shape[0]} sor x {df.shape[1]} oszlop")
print()
print(df.head(10).round(3))
print()
df

DataFrame: 300 sor x 6 oszlop

    xi_1   xi_2    eta  E_xi1_given_eta  E_xi2_given_eta  xi1_resid
0  2.993 -0.793  6.407            2.796           -0.788      0.198
1  5.046 -0.138  6.182            2.677           -0.833      2.369
2  5.158  1.105  5.948            2.554           -0.880      2.605
3  3.085 -1.149  4.865            1.983           -1.095      1.102
4  2.484 -3.112  2.724            0.854           -1.522      1.629
5  0.875 -2.636  4.840            1.970           -1.100     -1.094
6  0.184 -3.374  6.105            2.637           -0.848     -2.453
7  1.548 -1.088  2.763            0.875           -1.514      0.673
8  0.911 -1.275  2.833            0.912           -1.500     -0.001
9  2.751 -1.438  4.938            2.022           -1.081      0.730



,xi_1,xi_2,eta,E_xi1_given_eta,E_xi2_given_eta,xi1_resid
0,2.993428,-0.793237,6.406520,2.795529,-0.788142,0.197900
1,5.046060,-0.138468,6.182289,2.677342,-0.832805,2.368718
2,5.158426,1.104532,5.947569,2.553626,-0.879557,2.604800
3,3.085120,-1.148698,4.864879,1.982964,-1.095209,1.102157
4,2.483925,-3.112471,2.723832,0.854464,-1.521667,1.629461
...,...,...,...,...,...,...
295,5.693415,1.085136,6.363164,2.772677,-0.796778,2.920738
296,3.181310,0.772280,6.796723,3.001197,-0.710421,0.180113
297,3.014548,0.659354,7.204672,3.216218,-0.629165,-0.201669
298,4.764318,0.814395,6.172974,2.672432,-0.834661,2.091886
